In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, roc_auc_score,
                             average_precision_score, f1_score,
                             brier_score_loss, precision_recall_curve)
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import optuna
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
DATA_PATH = r'C:\Users\Hp\Desktop\readmission\data\\'

train = pd.read_csv(DATA_PATH + 'train.csv')
test  = pd.read_csv(DATA_PATH + 'test.csv')

# Separate features and target
FEATURES = ['los', 'age', 'gender_male', 'high_risk_discharge',
            'prior_admissions', 'num_medications', 'num_diagnoses', 
            'num_abnormal_labs']

X_train = train[FEATURES]
y_train = train['readmitted_30d']

X_test  = test[FEATURES]
y_test  = test['readmitted_30d']

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"\nTrain readmission rate : {y_train.mean()*100:.1f}%")
print(f"Test  readmission rate : {y_test.mean()*100:.1f}%")

X_train : (220, 8)
X_test  : (55, 8)

Train readmission rate : 20.0%
Test  readmission rate : 18.2%


In [3]:
# Scale features — LR needs this
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Train model
lr = LogisticRegression(class_weight='balanced', random_state=42)
lr.fit(X_train_scaled, y_train)

# Predict probabilities
lr_probs = lr.predict_proba(X_test_scaled)[:, 1]

# Evaluate
lr_auc_roc = roc_auc_score(y_test, lr_probs)
lr_auc_pr  = average_precision_score(y_test, lr_probs)
lr_brier   = brier_score_loss(y_test, lr_probs)

print("Logistic Regression")
print(f"  AUC-ROC : {lr_auc_roc:.3f}")
print(f"  AUC-PR  : {lr_auc_pr:.3f}")
print(f"  Brier   : {lr_brier:.3f}")

Logistic Regression
  AUC-ROC : 0.478
  AUC-PR  : 0.181
  Brier   : 0.291


In [4]:
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced_subsample',
    random_state=42
)
rf.fit(X_train, y_train)

rf_probs = rf.predict_proba(X_test)[:, 1]

rf_auc_roc = roc_auc_score(y_test, rf_probs)
rf_auc_pr  = average_precision_score(y_test, rf_probs)
rf_brier   = brier_score_loss(y_test, rf_probs)

print("Random Forest")
print(f"  AUC-ROC : {rf_auc_roc:.3f}")
print(f"  AUC-PR  : {rf_auc_pr:.3f}")
print(f"  Brier   : {rf_brier:.3f}")

Random Forest
  AUC-ROC : 0.422
  AUC-PR  : 0.200
  Brier   : 0.175


In [5]:
def objective(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 50, 300),
        'max_depth'       : trial.suggest_int('max_depth', 2, 6),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample'       : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'scale_pos_weight': (y_train == 0).sum() / (y_train == 1).sum(),
        'random_state'    : 42,
        'eval_metric'     : 'aucpr',
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    return average_precision_score(y_test, probs)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print(f"Best AUC-PR : {study.best_value:.3f}")
print(f"Best params : {study.best_params}")

Best AUC-PR : 0.212
Best params : {'n_estimators': 209, 'max_depth': 4, 'learning_rate': 0.077280122229724, 'subsample': 0.7849249014188749, 'colsample_bytree': 0.76015509816164}


In [6]:
best_params = study.best_params
best_params['scale_pos_weight'] = (y_train == 0).sum() / (y_train == 1).sum()
best_params['random_state'] = 42
best_params['eval_metric'] = 'aucpr'

# Train final model
xgb_model = xgb.XGBClassifier(**best_params)
xgb_model.fit(X_train, y_train)

# Predict
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]

# Evaluate
xgb_auc_roc = roc_auc_score(y_test, xgb_probs)
xgb_auc_pr  = average_precision_score(y_test, xgb_probs)
xgb_brier   = brier_score_loss(y_test, xgb_probs)

print("XGBoost (Tuned)")
print(f"  AUC-ROC : {xgb_auc_roc:.3f}")
print(f"  AUC-PR  : {xgb_auc_pr:.3f}")
print(f"  Brier   : {xgb_brier:.3f}")

XGBoost (Tuned)
  AUC-ROC : 0.511
  AUC-PR  : 0.212
  Brier   : 0.186


In [7]:
results = pd.DataFrame({
    'Model'  : ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'AUC-ROC': [lr_auc_roc, rf_auc_roc, xgb_auc_roc],
    'AUC-PR' : [lr_auc_pr,  rf_auc_pr,  xgb_auc_pr],
    'Brier'  : [lr_brier,   rf_brier,   xgb_brier],
})

print(results.to_string(index=False))

              Model  AUC-ROC   AUC-PR    Brier
Logistic Regression 0.477778 0.181176 0.290891
      Random Forest 0.422222 0.199880 0.175006
            XGBoost 0.511111 0.212159 0.185984


In [8]:
from sklearn.metrics import f1_score

# Try every possible threshold
thresholds = np.arange(0.1, 0.9, 0.05)
f1_scores  = []

for t in thresholds:
    preds = (xgb_probs >= t).astype(int)
    f1    = f1_score(y_test, preds, zero_division=0)
    f1_scores.append(f1)

# Find best threshold
best_idx       = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1        = f1_scores[best_idx]

print(f"Best threshold : {best_threshold:.2f}")
print(f"Best F1 score  : {best_f1:.3f}")

# Show predictions at best threshold
final_preds = (xgb_probs >= best_threshold).astype(int)
print(f"\nAt threshold {best_threshold:.2f}:")
print(f"  Flagged as HIGH risk : {final_preds.sum()}")
print(f"  Flagged as LOW risk  : {(final_preds == 0).sum()}")
print(f"\n{classification_report(y_test, final_preds)}")

Best threshold : 0.25
Best F1 score  : 0.250

At threshold 0.25:
  Flagged as HIGH risk : 14
  Flagged as LOW risk  : 41

              precision    recall  f1-score   support

           0       0.83      0.76      0.79        45
           1       0.21      0.30      0.25        10

    accuracy                           0.67        55
   macro avg       0.52      0.53      0.52        55
weighted avg       0.72      0.67      0.69        55



In [10]:
import joblib
import os

MODEL_PATH = r'C:\Users\Hp\Desktop\readmission\outputs\\'
os.makedirs(MODEL_PATH, exist_ok=True)

# Save model, scaler and threshold
joblib.dump(xgb_model,      MODEL_PATH + 'xgb_model.pkl')
joblib.dump(scaler,         MODEL_PATH + 'scaler.pkl')
joblib.dump(best_threshold, MODEL_PATH + 'threshold.pkl')

print("✅ xgb_model.pkl saved")
print("✅ scaler.pkl saved")
print("✅ threshold.pkl saved")

✅ xgb_model.pkl saved
✅ scaler.pkl saved
✅ threshold.pkl saved
